# Transformer Architectures -- Expert

> **Section 01** -- Topic 04 -- `expert` 
> **Prerequisites:** `advanced.ipynb` 
> **Objective:** Understand and implement Mixture of Experts (MoE) architectures -- routing mechanisms, load balancing, Switch Transformers, and scaling tradeoffs.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. Mixture of Experts (MoE)

The core idea behind MoE is simple: instead of one large FFN, use **many smaller expert FFNs** and a **router** that selects which experts process each token.

### Why MoE?

Standard (dense) transformers activate **all** parameters for every token. MoE activates only a subset:

- **Total parameters**: Very large (e.g., Mixtral 8x7B has ~46B total params)
- **Active parameters**: Only a fraction per token (e.g., Mixtral activates ~13B per token)
- **Result**: More knowledge stored, but same compute cost as a smaller dense model

### Architecture

```
Input token x
     |
  Router: softmax(x @ W_gate) -> select top-k experts
     |
  Expert 1: FFN_1(x) * gate_1
  Expert 3: FFN_3(x) * gate_3   (if top-2 routing)
     |
  Output = sum of weighted expert outputs
```

### Load balancing

Without regularization, the router tends to **collapse** -- sending all tokens to one or two experts. We need an auxiliary loss to encourage balanced routing.

In [ ]:
class Expert(nn.Module):
    """Single expert: a standard FFN."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)))


class TopKRouter(nn.Module):
    """Token-choice router with top-k expert selection and load balancing loss."""
    
    def __init__(self, d_model: int, n_experts: int, top_k: int = 2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, n_experts, bias=False)
    
    def forward(self, x):
        """Route tokens to experts.
        
        Args:
            x: (batch * seq_len, d_model) flattened token representations
        
        Returns:
            gate_scores: (batch * seq_len, top_k) - softmax weights for selected experts
            expert_indices: (batch * seq_len, top_k) - which experts to use
            aux_loss: load balancing loss
        """
        # Router logits
        logits = self.gate(x)  # (N, n_experts)
        
        # Top-k selection
        top_k_logits, top_k_indices = torch.topk(logits, self.top_k, dim=-1)
        
        # Softmax only over selected experts (not all experts)
        gate_scores = F.softmax(top_k_logits, dim=-1)
        
        # --- Load balancing auxiliary loss ---
        # f_i: fraction of tokens routed to expert i
        # p_i: average router probability for expert i
        # Loss = n_experts * sum(f_i * p_i)  (minimized when uniform)
        router_probs = F.softmax(logits, dim=-1)  # full softmax for loss computation
        
        # One-hot encode which experts were selected (across top_k)
        expert_mask = F.one_hot(top_k_indices, self.n_experts).float()  # (N, top_k, n_experts)
        expert_mask = expert_mask.sum(dim=1)  # (N, n_experts) - could be >1 if same expert selected twice
        
        # Fraction of tokens assigned to each expert
        f = expert_mask.mean(dim=0)  # (n_experts,)
        # Mean router probability per expert
        p = router_probs.mean(dim=0)  # (n_experts,)
        
        aux_loss = self.n_experts * (f * p).sum()
        
        return gate_scores, top_k_indices, aux_loss

In [ ]:
class MoELayer(nn.Module):
    """Mixture of Experts layer replacing the standard FFN."""
    
    def __init__(self, d_model: int, d_ff: int, n_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = TopKRouter(d_model, n_experts, top_k)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])
    
    def forward(self, x):
        """Forward pass with expert routing.
        
        Args:
            x: (B, T, d_model)
        Returns:
            output: (B, T, d_model)
            aux_loss: scalar load balancing loss
        """
        B, T, C = x.shape
        x_flat = x.view(-1, C)  # (B*T, C)
        
        # Route
        gate_scores, expert_indices, aux_loss = self.router(x_flat)
        # gate_scores: (B*T, top_k), expert_indices: (B*T, top_k)
        
        # Compute expert outputs (naive loop -- real implementations batch per expert)
        output = torch.zeros_like(x_flat)
        for i in range(self.top_k):
            expert_idx = expert_indices[:, i]  # (B*T,)
            gate_weight = gate_scores[:, i].unsqueeze(-1)  # (B*T, 1)
            
            for e in range(self.n_experts):
                mask = (expert_idx == e)  # which tokens go to expert e
                if mask.any():
                    expert_input = x_flat[mask]
                    expert_output = self.experts[e](expert_input)
                    output[mask] += gate_weight[mask] * expert_output
        
        return output.view(B, T, C), aux_loss


# Test MoE layer
d_model, d_ff = 128, 256
moe = MoELayer(d_model, d_ff, n_experts=8, top_k=2)
x = torch.randn(4, 32, d_model)

output, aux_loss = moe(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Aux loss:     {aux_loss.item():.4f}")
print(f"\nTotal MoE params:  {sum(p.numel() for p in moe.parameters()):,}")
print(f"Single expert FFN: {sum(p.numel() for p in moe.experts[0].parameters()):,}")
print(f"Active params per token (top-2 of 8): {2 * sum(p.numel() for p in moe.experts[0].parameters()):,}")

In [ ]:
# Visualize routing behavior
x = torch.randn(1, 128, d_model)  # 128 tokens
x_flat = x.view(-1, d_model)

with torch.no_grad():
    logits = moe.router.gate(x_flat)  # (128, 8)
    probs = F.softmax(logits, dim=-1)
    _, top_indices = torch.topk(probs, 2, dim=-1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Router probability heatmap
im = axes[0].imshow(probs.numpy(), aspect='auto', cmap='YlOrRd')
axes[0].set_xlabel('Expert')
axes[0].set_ylabel('Token')
axes[0].set_title('Router Probabilities (all experts)')
plt.colorbar(im, ax=axes[0])

# Expert load (how many tokens per expert)
expert_counts = torch.zeros(8)
for e in range(8):
    expert_counts[e] = (top_indices == e).sum().item()
axes[1].bar(range(8), expert_counts.numpy(), color='#2196F3')
axes[1].axhline(y=128 * 2 / 8, color='red', linestyle='--', label='Ideal (uniform)')
axes[1].set_xlabel('Expert')
axes[1].set_ylabel('Tokens Assigned')
axes[1].set_title('Expert Load Distribution (top-2 routing)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Per-token expert assignments
assignment_matrix = torch.zeros(128, 8)
for t in range(128):
    for k in range(2):
        assignment_matrix[t, top_indices[t, k]] = probs[t, top_indices[t, k]]
im = axes[2].imshow(assignment_matrix[:32].numpy(), aspect='auto', cmap='Blues')
axes[2].set_xlabel('Expert')
axes[2].set_ylabel('Token (first 32)')
axes[2].set_title('Selected Expert Weights (top-2)')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

---
## 2. Switch Transformer

The Switch Transformer (Fedus et al., 2021) simplifies MoE by using **top-1 routing**: each token goes to exactly one expert.

### Key innovations

1. **Top-1 routing**: Simpler, faster, and works surprisingly well
2. **Capacity factor**: Limits how many tokens each expert can process
3. **Selective precision**: Router in float32, experts in bfloat16

### Capacity factor

The **capacity** of each expert is:

$$C = \text{capacity\_factor} \times \frac{N}{E}$$

where $N$ = total tokens, $E$ = number of experts. If an expert exceeds capacity, overflow tokens are passed through with a residual connection (skipping the expert).

In [ ]:
class SwitchRouter(nn.Module):
    """Switch Transformer top-1 router with capacity factor."""
    
    def __init__(self, d_model: int, n_experts: int, capacity_factor: float = 1.25):
        super().__init__()
        self.n_experts = n_experts
        self.capacity_factor = capacity_factor
        self.gate = nn.Linear(d_model, n_experts, bias=False)
    
    def forward(self, x):
        """Top-1 routing with capacity limits.
        
        Args:
            x: (N, d_model) flattened tokens
        Returns:
            dispatch_mask: (N, n_experts) - binary, which expert gets each token
            gate_scores: (N,) - router confidence for the chosen expert
            aux_loss: load balancing loss
            overflow_mask: (N,) - True for tokens that exceeded expert capacity
        """
        N = x.shape[0]
        logits = self.gate(x)  # (N, n_experts)
        router_probs = F.softmax(logits, dim=-1)  # (N, n_experts)
        
        # Top-1 selection
        gate_scores, expert_indices = router_probs.max(dim=-1)  # (N,), (N,)
        
        # Capacity: max tokens per expert
        capacity = int(self.capacity_factor * N / self.n_experts)
        
        # Build dispatch mask with capacity enforcement
        dispatch_mask = torch.zeros(N, self.n_experts, device=x.device)
        overflow_mask = torch.zeros(N, dtype=torch.bool, device=x.device)
        expert_counts = torch.zeros(self.n_experts, dtype=torch.long, device=x.device)
        
        for i in range(N):
            e = expert_indices[i].item()
            if expert_counts[e] < capacity:
                dispatch_mask[i, e] = 1.0
                expert_counts[e] += 1
            else:
                overflow_mask[i] = True  # this token overflows
        
        # Load balancing loss
        f = dispatch_mask.sum(dim=0) / N  # fraction per expert
        p = router_probs.mean(dim=0)  # avg probability per expert
        aux_loss = self.n_experts * (f * p).sum()
        
        return dispatch_mask, gate_scores, aux_loss, overflow_mask, expert_counts

In [ ]:
class SwitchMoELayer(nn.Module):
    """Switch Transformer MoE layer with top-1 routing."""
    
    def __init__(self, d_model: int, d_ff: int, n_experts: int = 8,
                 capacity_factor: float = 1.25):
        super().__init__()
        self.router = SwitchRouter(d_model, n_experts, capacity_factor)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])
        self.n_experts = n_experts
    
    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        N = x_flat.shape[0]
        
        dispatch_mask, gate_scores, aux_loss, overflow_mask, expert_counts = self.router(x_flat)
        
        output = torch.zeros_like(x_flat)
        
        for e in range(self.n_experts):
            token_mask = dispatch_mask[:, e].bool()
            if token_mask.any():
                expert_input = x_flat[token_mask]
                expert_output = self.experts[e](expert_input)
                # Multiply by gate score
                output[token_mask] = gate_scores[token_mask].unsqueeze(-1) * expert_output
        
        # Overflow tokens: residual passthrough
        output[overflow_mask] = x_flat[overflow_mask]
        
        return output.view(B, T, C), aux_loss, overflow_mask.sum().item()


# Test Switch MoE
switch_moe = SwitchMoELayer(d_model=128, d_ff=256, n_experts=8, capacity_factor=1.25)
x = torch.randn(4, 32, 128)
out, aux_loss, n_overflow = switch_moe(x)
print(f"Output shape: {out.shape}")
print(f"Aux loss: {aux_loss.item():.4f}")
print(f"Overflow tokens: {n_overflow} / {4*32}")
print(f"Capacity per expert: {int(1.25 * 128 / 8)} tokens")

In [ ]:
# Visualize capacity factor effects
capacity_factors = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
overflow_counts = []
load_imbalances = []

x = torch.randn(1, 256, 128)  # 256 tokens

for cf in capacity_factors:
    switch = SwitchMoELayer(128, 256, n_experts=8, capacity_factor=cf)
    with torch.no_grad():
        _, _, n_ov = switch(x)
    overflow_counts.append(n_ov)
    
    # Check load balance
    x_flat = x.view(-1, 128)
    _, _, _, _, counts = switch.router(x_flat)
    ideal = 256 / 8
    imbalance = (counts.float() - ideal).abs().mean().item()
    load_imbalances.append(imbalance)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar([str(cf) for cf in capacity_factors], overflow_counts, color='#F44336')
axes[0].set_xlabel('Capacity Factor')
axes[0].set_ylabel('Overflow Tokens')
axes[0].set_title('Token Overflow vs Capacity Factor (256 tokens, 8 experts)')
axes[0].grid(True, alpha=0.3, axis='y')

# Show capacity limits
caps = [int(cf * 256 / 8) for cf in capacity_factors]
axes[1].bar([str(cf) for cf in capacity_factors], caps, color='#4CAF50', alpha=0.7, label='Capacity per expert')
axes[1].axhline(y=256/8, color='red', linestyle='--', label='Ideal (uniform)')
axes[1].set_xlabel('Capacity Factor')
axes[1].set_ylabel('Max Tokens per Expert')
axes[1].set_title('Expert Capacity')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 3. Expert Choice Routing

Expert Choice (Zhou et al., 2022) flips the routing: instead of tokens choosing experts, **experts choose tokens**.

### How it works

1. Compute affinity scores: $S = \text{softmax}(X \cdot W_g^T)^T$ -- shape `(n_experts, N)`
2. Each expert picks its top-k tokens from $S$
3. This guarantees **perfect load balancing**: every expert processes exactly $k$ tokens

### Tradeoff

- Advantage: No load balancing loss needed, guaranteed balance
- Disadvantage: Some tokens may be chosen by many experts (over-processed), others by none (under-processed)
- Disadvantage: Not autoregressive-friendly (needs to see all tokens)

In [ ]:
class ExpertChoiceRouter(nn.Module):
    """Expert Choice routing: experts choose their tokens."""
    
    def __init__(self, d_model: int, n_experts: int, tokens_per_expert: int = None):
        super().__init__()
        self.n_experts = n_experts
        self.tokens_per_expert = tokens_per_expert  # If None, computed as N * k / E
        self.gate = nn.Linear(d_model, n_experts, bias=False)
    
    def forward(self, x, top_k_tokens: int = None):
        """Route using expert choice.
        
        Args:
            x: (N, d_model) flattened tokens
            top_k_tokens: tokens each expert selects (default: 2 * N / n_experts)
        """
        N, C = x.shape
        if top_k_tokens is None:
            top_k_tokens = max(1, 2 * N // self.n_experts)
        
        # Compute scores: (N, n_experts)
        logits = self.gate(x)
        # Transpose and softmax along token dimension: each expert scores all tokens
        # scores: (n_experts, N)
        scores = F.softmax(logits.T, dim=-1)
        
        # Each expert picks top-k tokens
        # expert_gate: (n_experts, top_k_tokens) - scores for chosen tokens
        # expert_indices: (n_experts, top_k_tokens) - which tokens were chosen
        expert_gate, expert_indices = torch.topk(scores, top_k_tokens, dim=-1)
        
        return expert_gate, expert_indices


class ExpertChoiceMoELayer(nn.Module):
    """MoE layer with Expert Choice routing."""
    
    def __init__(self, d_model: int, d_ff: int, n_experts: int = 8):
        super().__init__()
        self.router = ExpertChoiceRouter(d_model, n_experts)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])
        self.n_experts = n_experts
    
    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)  # (N, C)
        N = x_flat.shape[0]
        
        expert_gate, expert_indices = self.router(x_flat)
        # expert_gate: (n_experts, k), expert_indices: (n_experts, k)
        
        output = torch.zeros_like(x_flat)
        
        for e in range(self.n_experts):
            # Gather tokens selected by this expert
            token_ids = expert_indices[e]  # (k,)
            gate_weights = expert_gate[e].unsqueeze(-1)  # (k, 1)
            
            expert_input = x_flat[token_ids]  # (k, C)
            expert_output = self.experts[e](expert_input)  # (k, C)
            
            # Accumulate (a token may be chosen by multiple experts)
            output.index_add_(0, token_ids, gate_weights * expert_output)
        
        # No auxiliary loss needed -- load is balanced by construction
        return output.view(B, T, C)


# Test
ec_moe = ExpertChoiceMoELayer(128, 256, n_experts=8)
x = torch.randn(4, 32, 128)
out = ec_moe(x)
print(f"Expert Choice MoE output shape: {out.shape}")
print(f"Tokens per expert: {2 * 128 // 8} (perfectly balanced!)")

In [ ]:
# Compare routing patterns: Token Choice vs Expert Choice
n_tokens = 128
n_experts = 8
d_model = 128

x = torch.randn(1, n_tokens, d_model)
x_flat = x.view(-1, d_model)

# Token Choice (top-2)
tc_router = TopKRouter(d_model, n_experts, top_k=2)
with torch.no_grad():
    tc_scores, tc_indices, _ = tc_router(x_flat)

# Expert Choice
ec_router = ExpertChoiceRouter(d_model, n_experts)
with torch.no_grad():
    ec_gate, ec_indices = ec_router(x_flat)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Token Choice: expert load
tc_counts = torch.zeros(n_experts)
for e in range(n_experts):
    tc_counts[e] = (tc_indices == e).sum().item()
axes[0].bar(range(n_experts), tc_counts.numpy(), color='#2196F3')
axes[0].axhline(y=n_tokens * 2 / n_experts, color='red', linestyle='--', label='Ideal')
axes[0].set_title('Token Choice: Expert Load')
axes[0].set_xlabel('Expert')
axes[0].set_ylabel('Tokens')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Expert Choice: expert load (always balanced!)
ec_counts = torch.tensor([ec_indices.shape[1]] * n_experts)
axes[1].bar(range(n_experts), ec_counts.numpy(), color='#4CAF50')
axes[1].axhline(y=n_tokens * 2 / n_experts, color='red', linestyle='--', label='Ideal')
axes[1].set_title('Expert Choice: Expert Load (guaranteed balanced)')
axes[1].set_xlabel('Expert')
axes[1].set_ylabel('Tokens')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Token coverage in Expert Choice: how many tokens were selected?
all_selected = ec_indices.flatten().unique()
never_selected = n_tokens - len(all_selected)
multi_selected = n_tokens - len(ec_indices.flatten().unique())

labels = ['Selected 1+ times', 'Never selected']
sizes = [len(all_selected), never_selected]
colors = ['#4CAF50', '#F44336']
axes[2].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[2].set_title(f'Expert Choice: Token Coverage\n({never_selected} tokens not processed by any expert)')

plt.tight_layout()
plt.show()

print(f"Token Choice load std: {tc_counts.std():.2f}")
print(f"Expert Choice load std: {ec_counts.float().std():.2f} (always 0)")
print(f"Expert Choice: {never_selected} tokens not selected by any expert")

---
## 4. Sparse vs Dense Tradeoffs

When does MoE actually help? Let us compare dense and sparse models matched on the same **compute budget** (FLOPs per forward pass).

### The key insight

MoE models have:
- **More total parameters** (stored in memory)
- **Same active parameters** per token (same compute)
- **Better performance per FLOP** (more knowledge per compute)

### When MoE wins

1. Training is **compute-bound** (not memory-bound)
2. Task benefits from **diverse specialization** (different experts for different patterns)
3. You can afford the **memory overhead** for storing all experts

### When dense wins

1. Deployment is **memory-constrained** (edge devices)
2. Batch sizes are **very small** (routing overhead dominates)
3. Task is **narrow** (does not benefit from expert specialization)

In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm for use in our comparison models."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


class DenseTransformerBlock(nn.Module):
    """Dense transformer block for comparison."""
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.d_model = d_model
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.norm2 = RMSNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.SiLU(),
            nn.Linear(d_ff, d_model, bias=False),
        )
    
    def _attn(self, x, mask):
        B, T, C = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)
    
    def forward(self, x, mask=None):
        x = x + self._attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class MoETransformerBlock(nn.Module):
    """MoE transformer block - replaces FFN with MoE."""
    def __init__(self, d_model, n_heads, d_ff, n_experts=8, top_k=2):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.d_model = d_model
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.norm2 = RMSNorm(d_model)
        self.moe = MoELayer(d_model, d_ff, n_experts, top_k)
    
    def _attn(self, x, mask):
        B, T, C = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)
    
    def forward(self, x, mask=None):
        x = x + self._attn(self.norm1(x), mask)
        normed = self.norm2(x)
        moe_out, aux_loss = self.moe(normed)
        x = x + moe_out
        return x, aux_loss

In [ ]:
# Build comparable dense and MoE models
d_model = 128
n_heads = 4

# Dense: larger FFN
dense_block = DenseTransformerBlock(d_model, n_heads, d_ff=512)

# MoE: 8 smaller experts, top-2 routing
# Each expert has d_ff=128, so active FFN params per token ~ 2 * 128 = 256
# vs dense 512 -- MoE uses LESS compute per token but more total params
moe_block = MoETransformerBlock(d_model, n_heads, d_ff=128, n_experts=8, top_k=2)

dense_total = sum(p.numel() for p in dense_block.parameters())
moe_total = sum(p.numel() for p in moe_block.parameters())

# Active params: for MoE, only top_k experts are activated
single_expert_ffn = sum(p.numel() for p in moe_block.moe.experts[0].parameters())
attn_params = sum(p.numel() for n, p in moe_block.named_parameters() if 'moe' not in n)
moe_active = attn_params + 2 * single_expert_ffn  # top-2

print(f"{'Metric':<25} {'Dense':>12} {'MoE (8 exp, top-2)':>18}")
print("-" * 58)
print(f"{'Total parameters':<25} {dense_total:>12,} {moe_total:>18,}")
print(f"{'Active params/token':<25} {dense_total:>12,} {moe_active:>18,}")
print(f"{'FFN hidden dim':<25} {'512':>12} {'128 x 8 experts':>18}")
print(f"{'Memory overhead':<25} {'1.0x':>12} {moe_total/dense_total:>17.1f}x")

In [ ]:
# Train both on same task and compare
VOCAB = 100
pattern = torch.tensor(list(range(1, 51)) * 40)

def get_batch(batch_size=32, seq_len=32):
    ix = torch.randint(0, len(pattern) - seq_len - 1, (batch_size,))
    x = torch.stack([pattern[i:i+seq_len] for i in ix])
    y = torch.stack([pattern[i+1:i+seq_len+1] for i in ix])
    return x, y

class SimpleModel(nn.Module):
    def __init__(self, block, vocab_size, d_model, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([block for _ in range(n_layers)])
        self.head = nn.Linear(d_model, vocab_size)
        self.is_moe = isinstance(block, MoETransformerBlock)
    
    def forward(self, x):
        h = self.embed(x)
        T = x.size(1)
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        total_aux = 0
        for block in self.blocks:
            if self.is_moe:
                h, aux = block(h, mask)
                total_aux += aux
            else:
                h = block(h, mask)
        return self.head(h), total_aux if self.is_moe else 0

def train_model(model, name, n_steps=500, aux_weight=0.01):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    losses = []
    for step in range(n_steps):
        x, y = get_batch()
        logits, aux = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss = loss + aux_weight * aux if isinstance(aux, torch.Tensor) else loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if step % 100 == 0:
            print(f"  {name} step {step}: loss={loss.item():.4f}")
    return losses

d_model = 64
dense_model = SimpleModel(
    DenseTransformerBlock(d_model, 4, 256), VOCAB, d_model, n_layers=1
)
moe_model = SimpleModel(
    MoETransformerBlock(d_model, 4, 64, n_experts=8, top_k=2), VOCAB, d_model, n_layers=1
)

print("Training Dense model:")
dense_losses = train_model(dense_model, "Dense")
print("\nTraining MoE model:")
moe_losses = train_model(moe_model, "MoE")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dense_losses, label=f'Dense ({sum(p.numel() for p in dense_model.parameters()):,} params)', alpha=0.7)
ax.plot(moe_losses, label=f'MoE ({sum(p.numel() for p in moe_model.parameters()):,} params)', alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Dense vs MoE Training Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Architecture Scaling

When designing a transformer, you must decide: **how deep, how wide, how many heads, what FFN ratio?** These decisions interact in complex ways.

### Key rules of thumb

| Decision | Rule of thumb | Reasoning |
|----------|--------------|----------|
| **Depth vs Width** | Scale both together | Depth helps reasoning, width helps knowledge |
| **d_model** | Usually 64 * n_heads | Keep d_k = 64 or 128 |
| **n_heads** | d_model / 64 or d_model / 128 | Each head needs enough dimensions to be useful |
| **d_ff** | 4x d_model (classic) or 8/3x (SwiGLU) | Roughly 2/3 of block params in FFN |
| **n_layers** | More layers for reasoning tasks | Diminishing returns past ~80 layers |
| **KV heads** | n_heads / 4 to n_heads / 8 for GQA | Balances quality and inference memory |

### Scaling laws

Kaplan et al. (2020) and Hoffmann et al. (2022, "Chinchilla") showed that loss scales as a power law with compute, data, and parameters. The Chinchilla-optimal allocation is roughly:

$$N_{\text{opt}} \propto C^{0.5}, \quad D_{\text{opt}} \propto C^{0.5}$$

i.e., for a given compute budget $C$, scale parameters $N$ and training tokens $D$ equally.

In [ ]:
def count_transformer_params(vocab_size, d_model, n_layers, n_heads,
                              d_ff=None, n_kv_heads=None, has_bias=False):
    """Count parameters in a decoder-only transformer."""
    if d_ff is None:
        d_ff = 4 * d_model
    if n_kv_heads is None:
        n_kv_heads = n_heads
    d_k = d_model // n_heads
    
    # Embedding
    embed = vocab_size * d_model
    # Position embedding (if learned)
    # pos_embed = max_len * d_model  # skip, RoPE has none
    
    # Per-layer attention
    q_params = d_model * (n_heads * d_k)
    k_params = d_model * (n_kv_heads * d_k)
    v_params = d_model * (n_kv_heads * d_k)
    o_params = (n_heads * d_k) * d_model
    attn_per_layer = q_params + k_params + v_params + o_params
    
    # Per-layer FFN
    ffn_per_layer = 2 * d_model * d_ff
    if has_bias:
        ffn_per_layer += d_ff + d_model
    
    # Norms (2 per layer + 1 final)
    norm_per_layer = 2 * d_model
    final_norm = d_model
    
    total_per_layer = attn_per_layer + ffn_per_layer + norm_per_layer
    total = embed + n_layers * total_per_layer + final_norm
    # LM head is typically tied to embedding, so no extra params
    
    return {
        'embedding': embed,
        'attention_per_layer': attn_per_layer,
        'ffn_per_layer': ffn_per_layer,
        'norm_per_layer': norm_per_layer,
        'total_per_layer': total_per_layer,
        'total': total,
    }


# Compare famous architectures
architectures = {
    'GPT-2 Small': dict(vocab_size=50257, d_model=768, n_layers=12, n_heads=12, d_ff=3072),
    'GPT-2 Medium': dict(vocab_size=50257, d_model=1024, n_layers=24, n_heads=16, d_ff=4096),
    'GPT-2 Large': dict(vocab_size=50257, d_model=1280, n_layers=36, n_heads=20, d_ff=5120),
    'Llama 3 8B': dict(vocab_size=128256, d_model=4096, n_layers=32, n_heads=32, d_ff=14336, n_kv_heads=8),
    'Llama 3 70B': dict(vocab_size=128256, d_model=8192, n_layers=80, n_heads=64, d_ff=28672, n_kv_heads=8),
}

print(f"{'Model':<18} {'Total Params':>14} {'Embed%':>8} {'Attn%':>8} {'FFN%':>8} {'d_model':>8} {'Layers':>7}")
print("-" * 80)
for name, config in architectures.items():
    info = count_transformer_params(**config)
    t = info['total']
    e_pct = info['embedding'] / t * 100
    a_pct = info['attention_per_layer'] * config['n_layers'] / t * 100
    f_pct = info['ffn_per_layer'] * config['n_layers'] / t * 100
    
    if t > 1e9:
        param_str = f"{t/1e9:.1f}B"
    else:
        param_str = f"{t/1e6:.0f}M"
    
    print(f"{name:<18} {param_str:>14} {e_pct:>7.1f}% {a_pct:>7.1f}% {f_pct:>7.1f}% {config['d_model']:>8} {config['n_layers']:>7}")

In [ ]:
# Visualize how parameter count scales with different dimensions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Depth scaling (fixed width)
layers_range = range(1, 101)
params_by_depth = [count_transformer_params(32000, 1024, n, 16)['total'] / 1e6 for n in layers_range]
axes[0].plot(list(layers_range), params_by_depth, linewidth=2)
axes[0].set_xlabel('Number of Layers')
axes[0].set_ylabel('Parameters (M)')
axes[0].set_title('Scaling by Depth (d=1024)')
axes[0].grid(True, alpha=0.3)

# 2. Width scaling (fixed depth)
widths = [128, 256, 512, 768, 1024, 1536, 2048, 3072, 4096]
params_by_width = [count_transformer_params(32000, d, 12, d//64)['total'] / 1e6 for d in widths]
axes[1].plot(widths, params_by_width, 'o-', linewidth=2)
axes[1].set_xlabel('d_model')
axes[1].set_ylabel('Parameters (M)')
axes[1].set_title('Scaling by Width (12 layers)')
axes[1].grid(True, alpha=0.3)

# 3. Depth vs Width for fixed param budget
# For ~100M params, what's the optimal depth/width split?
target = 100  # million params
configs = []
for n_layers in [4, 6, 8, 12, 16, 24, 32, 48]:
    for d_model in [128, 192, 256, 384, 512, 768, 1024, 1536, 2048]:
        if d_model % 64 != 0:
            continue
        n_heads = d_model // 64
        if n_heads < 1:
            continue
        p = count_transformer_params(32000, d_model, n_layers, n_heads)['total'] / 1e6
        if 50 < p < 200:  # within 2x of target
            configs.append((n_layers, d_model, p))

layers_arr = [c[0] for c in configs]
widths_arr = [c[1] for c in configs]
params_arr = [c[2] for c in configs]

sc = axes[2].scatter(layers_arr, widths_arr, c=params_arr, cmap='viridis', s=50, alpha=0.8)
plt.colorbar(sc, ax=axes[2], label='Params (M)')
axes[2].set_xlabel('Number of Layers')
axes[2].set_ylabel('d_model')
axes[2].set_title('Depth/Width Configurations (50-200M params)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Chinchilla optimal scaling visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Compute budget in FLOPs (approximate: 6 * N * D for forward + backward)
compute_budgets = np.logspace(18, 25, 50)  # 1e18 to 1e25 FLOPs

# Chinchilla-optimal: N_opt ~ C^0.5, D_opt ~ C^0.5
# More precisely: N_opt = a * C^alpha, D_opt = b * C^beta
# with alpha ~ 0.50, beta ~ 0.50 (Hoffmann et al.)
# Using 6*N*D = C: N = sqrt(C/6), D = sqrt(C/6)
chinchilla_N = np.sqrt(compute_budgets / 6)
chinchilla_D = np.sqrt(compute_budgets / 6)

# Kaplan (earlier) scaling: train bigger model, fewer tokens
# N ~ C^0.73, D ~ C^0.27
kaplan_N = compute_budgets ** 0.73 / (6 * compute_budgets ** 0.27)
kaplan_N = np.clip(kaplan_N, 1e6, 1e13)

ax.loglog(compute_budgets, chinchilla_N, label='Chinchilla-optimal N', linewidth=2)
ax.loglog(compute_budgets, chinchilla_D, label='Chinchilla-optimal D', linewidth=2, linestyle='--')

# Mark some real models
models_data = {
    'GPT-3 175B': (3.6e23, 175e9, 300e9),  # (compute, params, tokens)
    'Chinchilla 70B': (5.8e23, 70e9, 1.4e12),
    'Llama 2 70B': (1e24, 70e9, 2e12),
    'Llama 3 8B': (3e23, 8e9, 15e12),
}

for name, (c, n, d) in models_data.items():
    ax.scatter(c, n, s=100, zorder=5, marker='o')
    ax.annotate(name, (c, n), textcoords="offset points", xytext=(10, 5), fontsize=9)

ax.set_xlabel('Compute Budget (FLOPs)', fontsize=12)
ax.set_ylabel('Parameters / Tokens', fontsize=12)
ax.set_title('Scaling Laws: Optimal Model Size vs Compute', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Key insight: Llama 3 8B was trained on 15T tokens -- far more than Chinchilla-optimal.")
print("This 'over-training' on more data yields better models for a given inference cost.")

---
## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **MoE with Top-K** | Route each token to K experts; needs load balancing loss |
| **Switch Transformer** | Top-1 routing + capacity factor; simpler and fast |
| **Expert Choice** | Experts pick tokens; perfect balance, but some tokens may be missed |
| **Sparse vs Dense** | MoE: more total params, same active compute; wins when compute-bound |
| **Scaling** | Width scales as O(d^2), depth scales linearly. Chinchilla: scale N and D equally. |

### Next: `build.ipynb`

We put everything together and build a complete GPT from scratch, adding modern improvements, training it on real text, and comparing variants.